![image.png](https://i.imgur.com/a3uAqnb.png)

# Stereo Matching with a Learned Cost Volume on KITTI 2015

- **Dataset**: [KITTI Stereo 2015](https://www.kaggle.com/datasets/sadeghkalami/kitti-stereo-evaluation-2015) from Kaggle, 200 rectified street-scene stereo pairs with LiDAR ground-truth disparity
- **Task**: stereo matching (a disparity for every pixel) and metric depth from disparity
- **Model**: a small GC-Net-style network (about 0.8M parameters) trained from scratch, against OpenCV's semi-global matching
- **Runtime**: Colab T4 GPU, about 7 minutes of training

In this lab, we will:

- score a classical matcher, semi-global matching, by end-point error and KITTI's D1 outlier rate
- build a cost volume from learned features and regularise it with 3D convolutions
- read out sub-pixel disparities with the soft argmin and train the network end to end
- turn disparities into depth and a 3D point cloud, and see why far points are less precise

## Stereo matching in brief

Two cameras a baseline $B$ apart see a scene from two viewpoints. After rectification, a warp that aligns their image rows, a scene point lands on the same row in both images, at column $u_L$ on the left and $u_R$ on the right. Their difference $d = u_L - u_R$ is the **disparity**, and similar triangles give the depth:

$$Z = \frac{fB}{d}$$

where $Z$ is the depth in metres, $f$ the focal length in pixels and $B$ the baseline in metres.

A matcher scores each left pixel $(u, v)$ against the right pixels $(u - d, v)$ for the candidates $d = 0, \dots, D-1$. The scores of an $H \times W$ image form an $H \times W \times D$ **cost volume**, classically read out by taking each pixel's lowest cost (winner-take-all). GC-Net (Kendall et al., 2017) learns the costs: a CNN computes features of both images, 3D convolutions turn the stacked features into one cost $c_d$ per pixel and disparity, and a **soft argmin** gives the disparity:

$$\hat{d} = \sum_{d=0}^{D-1} d \cdot \sigma_d(-\mathbf{c})$$

Where:
- $\mathbf{c} = (c_0, \dots, c_{D-1})$ are the costs of one pixel, low for a good match
- $\sigma_d(-\mathbf{c}) = e^{-c_d} / \sum_k e^{-c_k}$ is the softmax, so $\hat{d}$ is the expected disparity

Being differentiable and sub-pixel, the soft argmin lets an L1 loss on the pixels with ground truth train the whole network.

Paper: [Kendall et al., 2017. End-to-End Learning of Geometry and Context for Deep Stereo Regression](https://arxiv.org/abs/1703.04309)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
import os
import random
import time

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
N_TRAIN = 160           # training pairs; KITTI 2015 has 200 pairs with ground truth
N_VAL = 40              # held-out pairs for validation, as in GC-Net
MAX_DISP = 192          # D: candidate disparities 0..191 px, as in GC-Net and PSMNet; a multiple of 16
CROP_H = 256            # training crop height
CROP_W = 512            # training crop width
FEAT_CH = 32            # channels of the 2D feature network
COST_CH = 32            # channels of the 3D cost network
BATCH_SIZE = 4
EPOCHS = 30             # one epoch = one random crop from every training pair
LR = 1e-3
EVAL_EVERY = 5          # epochs between validation passes
SGBM_BLOCK = 5          # block size of semi-global matching
NUM_WORKERS = 2         # DataLoader worker processes; a Colab runtime has 2 CPU cores

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True   # input sizes repeat, so cuDNN can pick the fastest 3D convolutions

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | OpenCV {cv2.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

## 1. Dataset

- **Source**: [KITTI Stereo 2015](https://www.cvlibs.net/datasets/kitti/eval_scene_flow.php?benchmark=stereo) (Menze and Geiger, 2015), Kaggle mirror version 1 (1.7 GB)
- **Content**: 200 training scenes filmed from a car around Karlsruhe, each a rectified left (`image_2`) and right (`image_3`) image of about $1242 \times 375$ pixels
- **Ground truth**: `disp_occ_0`, the disparity of the left image from accumulated LiDAR scans and fitted 3D car models, so it is sparse; a 16-bit PNG where value / 256 is the disparity and 0 means no measurement
- **Licence**: CC BY-NC-SA 3.0 (KITTI); the mirror lists Apache 2.0
- **What we use**: frame 10 of the 200 training scenes, 160 for training and 40 for validation; the test scenes have no public ground truth

Expected time: about a minute to download and unpack.

In [ ]:
path = kagglehub.dataset_download("sadeghkalami/kitti-stereo-evaluation-2015/versions/1")
DATA_DIR = os.path.join(path, "training")
print("Path to dataset files:", path)


def read_pair(scene_id):
    """Left and right images (H, W, 3) uint8 and the ground-truth disparity (H, W) float32 of one scene."""
    left = np.array(Image.open(os.path.join(DATA_DIR, "image_2", f"{scene_id}_10.png")).convert("RGB"))
    right = np.array(Image.open(os.path.join(DATA_DIR, "image_3", f"{scene_id}_10.png")).convert("RGB"))
    disp = np.asarray(Image.open(os.path.join(DATA_DIR, "disp_occ_0", f"{scene_id}_10.png"))).astype(np.float32) / 256.0
    return left, right, disp


scene_ids = sorted(name[:6] for name in os.listdir(os.path.join(DATA_DIR, "disp_occ_0")))
left, right, disp_gt = read_pair(scene_ids[0])
has_gt = disp_gt > 0
print(f"{len(scene_ids)} scenes | scene {scene_ids[0]}: image {left.shape}, disparity {disp_gt.shape}")
print(f"Pixels with ground truth: {has_gt.mean():.1%} | disparities from {disp_gt[has_gt].min():.1f} to {disp_gt[has_gt].max():.1f} px")

In the overlay (red from the left image, green and blue from the right) every point shifts only horizontally: a lot for nearby cars and trees, hardly at all at the far end of the street. The bottom panel draws the LiDAR disparities.

In [ ]:
anaglyph = np.dstack([left[..., 0], right[..., 1], right[..., 2]])
ys, xs = np.nonzero(has_gt)

fig, axes = plt.subplots(3, 1, figsize=(12, 10.5))
axes[0].imshow(left)
axes[0].set_title("Left image")
axes[1].imshow(anaglyph)
axes[1].set_title("Left (red) over right (cyan): near objects shift further")
axes[2].imshow(left)
points = axes[2].scatter(xs, ys, c=disp_gt[ys, xs], s=0.3, cmap="magma")
axes[2].set_title("Ground-truth disparity")
fig.colorbar(points, ax=axes[2], fraction=0.02, pad=0.01, label="disparity (px)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Depth needs only $f$ and $B$; placing a pixel in 3D also needs the principal point $(c_x, c_y)$, where the optical axis meets the image. The mirror has no calibration files, so we copy KITTI's values; each recording day has its own calibration and image width. A matching error $\Delta d$ moves the depth by $\Delta Z \approx Z^2 \Delta d / (fB)$, the derivative of $Z = fB/d$, so it grows with the square of the distance.

In [ ]:
# Colour camera pair of each KITTI recording day: image width (px) -> (f px, B m, c_x px, c_y px),
# from P_rect_02 and P_rect_03 in the KITTI calibration files (calib_cam_to_cam.txt).
CALIBRATION = {
    1242: (721.5377, 0.5327, 609.5593, 172.8540),   # 2011-09-26
    1241: (718.8560, 0.5323, 607.1928, 185.2157),   # 2011-10-03
    1238: (718.3351, 0.5301, 600.3891, 181.5122),   # 2011-09-29
    1224: (707.0493, 0.5373, 604.0814, 180.5066),   # 2011-09-28
}

f_px, baseline, _, _ = CALIBRATION[left.shape[1]]
print(f"Scene {scene_ids[0]}: f = {f_px:.1f} px, B = {baseline:.3f} m")
for depth in [5, 10, 20, 40, 80]:
    print(f"Z = {depth:>2} m: d = {f_px * baseline / depth:5.1f} px, and a 1 px error moves Z by {depth ** 2 / (f_px * baseline):5.2f} m")

We shuffle the scenes with the seed, keep the last 40 for validation (the same scenes whatever `N_TRAIN` is) and the first 160 for training, and hold them in memory (about 1 GB). Training uses random $256 \times 512$ crops, as GC-Net does, cut at the same place in both images so that disparities stay unchanged, and no flips, since a mirrored left image would act as a right one.

In [ ]:
order = np.random.default_rng(SEED).permutation(len(scene_ids))
val_ids = [scene_ids[i] for i in order[len(order) - N_VAL:]]               # the same scenes whatever N_TRAIN is
train_ids = [scene_ids[i] for i in order[:len(order) - N_VAL][:N_TRAIN]]   # never overlaps them
train_pairs = [read_pair(s) for s in tqdm(train_ids, desc="train scenes")]
val_pairs = [read_pair(s) for s in tqdm(val_ids, desc="validation scenes")]


class StereoCrops(Dataset):
    """One random CROP_H x CROP_W window per pair, cut at the same place in both images and the disparity."""

    def __init__(self, pairs, crop_h, crop_w):
        self.pairs = pairs
        self.crop_h = crop_h
        self.crop_w = crop_w

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        left, right, disp = self.pairs[i]
        y = random.randint(0, disp.shape[0] - self.crop_h)
        x = random.randint(0, disp.shape[1] - self.crop_w)
        rows, cols = slice(y, y + self.crop_h), slice(x, x + self.crop_w)
        left_crop, right_crop = [torch.from_numpy(np.ascontiguousarray(img[rows, cols].transpose(2, 0, 1)))
                                 for img in (left, right)]
        return left_crop, right_crop, torch.from_numpy(disp[rows, cols].copy())


train_loader = DataLoader(StereoCrops(train_pairs, CROP_H, CROP_W), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True, pin_memory=DEVICE == "cuda")
left_b, right_b, disp_b = next(iter(train_loader))
print(f"{len(train_pairs)} training and {len(val_pairs)} validation pairs")
print(left_b.shape, left_b.dtype, disp_b.shape)   # (B, 3, CROP_H, CROP_W) uint8, (B, CROP_H, CROP_W)

## 2. A classical baseline: semi-global matching

OpenCV's `StereoSGBM` implements semi-global matching (Hirschmüller, 2008): a cost comparing small pixel blocks for every candidate disparity, smoothed along 5 image directions with a penalty $P_1$ for one-pixel disparity steps and a larger $P_2$ for bigger jumps, then winner-take-all. It never searches the first $D$ columns, whose candidates would run off the right image, and post-filters reject doubtful matches. KITTI fills such holes before scoring with the smaller (background) of the nearest valid disparities on the row; so do we. We score every method over all ground-truth pixels:

- **EPE** (end-point error): the mean $|\hat{d} - d|$ in pixels
- **D1**: the KITTI 2015 benchmark's metric, the share of pixels whose error exceeds both 3 px and 5% of the true disparity

Expected time: under a minute.

In [ ]:
sgbm = cv2.StereoSGBM.create(
    minDisparity=0,
    numDisparities=MAX_DISP,                  # must be a multiple of 16
    blockSize=SGBM_BLOCK,
    P1=8 * 3 * SGBM_BLOCK ** 2,               # values suggested by the OpenCV documentation for colour images
    P2=32 * 3 * SGBM_BLOCK ** 2,
    uniquenessRatio=10,                       # the best cost must beat the second best by 10%
    speckleWindowSize=100,                    # remove small isolated blobs
    speckleRange=2,
    disp12MaxDiff=1,                          # left-right consistency check, in pixels
)


def sgbm_disparity(left, right):
    """Semi-global matching on one pair; NaN where OpenCV gives no disparity (left border, rejected matches)."""
    disp = sgbm.compute(left, right).astype(np.float32) / 16.0   # OpenCV returns 16 x disparity as int16
    disp[disp < 0] = np.nan
    return disp


def fill_holes(disp):
    """Fill NaNs row by row with the smaller of the nearest valid disparities to the left and right."""
    h, w = disp.shape
    valid = ~np.isnan(disp)
    cols = np.arange(w)
    left_idx = np.maximum.accumulate(np.where(valid, cols, -1), axis=1)                    # last valid column <= u
    right_idx = np.minimum.accumulate(np.where(valid, cols, w)[:, ::-1], axis=1)[:, ::-1]  # first valid column >= u
    rows = np.arange(h)[:, None]
    from_left = np.where(left_idx >= 0, disp[rows, left_idx.clip(0, w - 1)], np.inf)
    from_right = np.where(right_idx < w, disp[rows, right_idx.clip(0, w - 1)], np.inf)
    filled = np.minimum(from_left, from_right)
    filled[np.isinf(filled)] = 0.0                                                         # a row with no valid pixel
    return np.where(valid, disp, filled)


def score(preds, pairs):
    """EPE (px) and D1 (fraction) over every ground-truth pixel, pooled over pixels as KITTI does."""
    abs_sum, n_outliers, n_pixels = 0.0, 0, 0
    for pred, (_, _, gt) in zip(preds, pairs):
        mask = gt > 0
        err = np.abs(pred[mask] - gt[mask])
        abs_sum += err.sum()
        n_outliers += np.count_nonzero((err > 3.0) & (err > 0.05 * gt[mask]))
        n_pixels += err.size
    return abs_sum / n_pixels, n_outliers / n_pixels


start = time.time()
sgbm_raw = [sgbm_disparity(left, right) for left, right, _ in tqdm(val_pairs, desc="SGBM")]
sgbm_seconds = (time.time() - start) / len(val_pairs)
sgbm_preds = [fill_holes(d) for d in sgbm_raw]
sgbm_epe, sgbm_d1 = score(sgbm_preds, val_pairs)
rejected = np.mean([np.isnan(d[:, MAX_DISP:]).mean() for d in sgbm_raw])   # the first MAX_DISP columns are never searched
print(f"SGBM: EPE {sgbm_epe:.2f} px | D1 {sgbm_d1:.1%} | rejected {rejected:.1%} of the searched pixels | "
      f"{sgbm_seconds:.2f} s per pair")

## 3. Model: a small GC-Net

GC-Net's four stages, scaled down for a T4:

1. **Features**: one 2D CNN, shared by both images so that their features are comparable, at a quarter of the resolution.
2. **Cost volume**: the left feature at $(u, v)$ next to the right feature at $(u - d, v)$ for each candidate $d$. We stack the right feature map shifted by 0, 1, 2, ... columns on a disparity axis and zero the columns without a partner. A quarter-resolution column is 4 pixels, so $D/4 = 48$ shifts cover 0 to 188 px.
3. **Regularisation**: 3D convolutions over disparity, height and width, with an encoder-decoder for wider context, give one cost per disparity and pixel.
4. **Soft argmin**, after interpolating the costs back to full resolution and all $D = 192$ candidates.

The quarter-resolution volume follows PSMNet (Chang and Chen, 2018); GC-Net works at half resolution.

In [ ]:
def conv_bn(in_ch, out_ch, kernel=3, stride=1, relu=True):
    """2D convolution, batch norm and (optionally) ReLU."""
    layers = [nn.Conv2d(in_ch, out_ch, kernel, stride, kernel // 2, bias=False), nn.BatchNorm2d(out_ch)]
    if relu:
        layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)


class ResBlock(nn.Module):
    """Two 3x3 convolutions with a skip connection."""

    def __init__(self, ch):
        super().__init__()
        self.body = nn.Sequential(conv_bn(ch, ch), conv_bn(ch, ch, relu=False))

    def forward(self, x):
        return F.relu(x + self.body(x))


class FeatureNet(nn.Module):
    """Image (B, 3, H, W) in [-1, 1] -> features (B, C, H/4, W/4)."""

    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            conv_bn(3, ch, kernel=5, stride=2),      # (B, C, H/2, W/2)
            ResBlock(ch), ResBlock(ch),
            conv_bn(ch, ch, stride=2),               # (B, C, H/4, W/4)
            ResBlock(ch), ResBlock(ch), ResBlock(ch),
            nn.Conv2d(ch, ch, 3, 1, 1),              # no batch norm or ReLU on the output, as in GC-Net
        )

    def forward(self, x):
        return self.net(x)


def build_cost_volume(feat_left, feat_right, num_disp):
    """Concatenation cost volume (B, 2C, num_disp, H, W) from left and right features (B, C, H, W)."""
    w = feat_left.shape[-1]
    shifted = [F.pad(feat_right[..., : w - d], (d, 0)) for d in range(num_disp)]   # column u holds right feature u - d
    right_volume = torch.stack(shifted, dim=2)                                        # (B, C, D, H, W)
    left_volume = feat_left.unsqueeze(2).expand(-1, -1, num_disp, -1, -1)             # (B, C, D, H, W)
    has_partner = torch.arange(w, device=feat_left.device) >= torch.arange(num_disp, device=feat_left.device)[:, None]
    mask = has_partner.view(1, 1, num_disp, 1, w).to(feat_left.dtype)                 # 1 where u - d >= 0
    return torch.cat([left_volume, right_volume], dim=1) * mask

The 3D part halves disparity, height and width twice and comes back up with transposed convolutions and skip connections. `costs` returns the full-resolution costs in float32, so mixed precision never rounds the probabilities, and `forward` applies the softmax and the soft argmin.

In [ ]:
def conv3d_bn(in_ch, out_ch, stride=1):
    """3D convolution over (disparity, height, width), batch norm and ReLU."""
    return nn.Sequential(nn.Conv3d(in_ch, out_ch, 3, stride, 1, bias=False), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))


class Hourglass3D(nn.Module):
    """3D encoder-decoder with two stride-2 levels and skip connections."""

    def __init__(self, ch):
        super().__init__()
        self.down1 = nn.Sequential(conv3d_bn(ch, 2 * ch, stride=2), conv3d_bn(2 * ch, 2 * ch))
        self.down2 = nn.Sequential(conv3d_bn(2 * ch, 2 * ch, stride=2), conv3d_bn(2 * ch, 2 * ch))
        self.up1 = nn.Sequential(nn.ConvTranspose3d(2 * ch, 2 * ch, 3, 2, 1, output_padding=1, bias=False),
                                 nn.BatchNorm3d(2 * ch))
        self.up2 = nn.Sequential(nn.ConvTranspose3d(2 * ch, ch, 3, 2, 1, output_padding=1, bias=False),
                                 nn.BatchNorm3d(ch))

    def forward(self, x):                  # x: (B, ch, D/4, H/4, W/4)
        d1 = self.down1(x)                 # (B, 2ch, D/8, H/8, W/8)
        d2 = self.down2(d1)                # (B, 2ch, D/16, H/16, W/16)
        u1 = F.relu(self.up1(d2) + d1)     # (B, 2ch, D/8, H/8, W/8)
        return F.relu(self.up2(u1) + x)    # (B, ch, D/4, H/4, W/4)


def soft_argmin(prob):
    """Expected disparity sum_d d * p_d of probabilities (B, D, H, W) -> (B, H, W)."""
    disparities = torch.arange(prob.shape[1], device=prob.device, dtype=prob.dtype).view(1, -1, 1, 1)
    return (prob * disparities).sum(dim=1)


class MiniGCNet(nn.Module):
    """Stereo pair (B, 3, H, W) x 2 -> disparity (B, H, W); H and W must be multiples of 16."""

    def __init__(self, max_disp, feat_ch, cost_ch):
        super().__init__()
        self.max_disp = max_disp
        self.features = FeatureNet(feat_ch)
        self.regularise = nn.Sequential(
            conv3d_bn(2 * feat_ch, cost_ch), conv3d_bn(cost_ch, cost_ch),
            Hourglass3D(cost_ch),
            conv3d_bn(cost_ch, cost_ch),
            nn.Conv3d(cost_ch, 1, 3, 1, 1),                                     # one cost per disparity and pixel
        )

    def costs(self, left, right):
        h, w = left.shape[-2:]
        feat_left, feat_right = self.features(left), self.features(right)       # (B, C, H/4, W/4) each
        volume = build_cost_volume(feat_left, feat_right, self.max_disp // 4)   # (B, 2C, D/4, H/4, W/4)
        cost = self.regularise(volume)                                          # (B, 1, D/4, H/4, W/4)
        cost = F.interpolate(cost, size=(self.max_disp, h, w), mode="trilinear", align_corners=False)
        return cost.squeeze(1).float()                                          # (B, D, H, W)

    def forward(self, left, right):
        prob = F.softmax(-self.costs(left, right), dim=1)                       # (B, D, H, W), sums to 1 over D
        return soft_argmin(prob)                                                # (B, H, W)


model = MiniGCNet(MAX_DISP, FEAT_CH, COST_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

For full images, `disparity_probabilities` scales pixels to $[-1, 1]$ as GC-Net does, pads the image at the top and right to a multiple of 16 (the network halves the size four times) and crops back; `predict` takes the soft argmin, or the winner-take-all argmax with `hard=True`. Untrained, all costs are nearly equal, so the soft argmin gives about the mean candidate, $(D-1)/2 = 95.5$ px.

In [ ]:
def to_input(images):
    """uint8 (B, 3, H, W) -> float in [-1, 1] on DEVICE."""
    return images.to(DEVICE, non_blocking=True).float() / 127.5 - 1.0


@torch.no_grad()
def disparity_probabilities(model, left, right):
    """Softmax over the disparities (D, H, W) for one uint8 (H, W, 3) pair, on DEVICE."""
    model.eval()
    h, w = left.shape[:2]
    pad_h, pad_w = -h % 16, -w % 16
    left_t, right_t = [F.pad(to_input(torch.from_numpy(img).permute(2, 0, 1)[None]), (0, pad_w, pad_h, 0))
                       for img in (left, right)]
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        cost = model.costs(left_t, right_t)
    return F.softmax(-cost, dim=1)[0, :, pad_h:, :w]


def predict(model, left, right, hard=False):
    """Disparity map (H, W) by soft argmin, or by winner-take-all when hard=True."""
    prob = disparity_probabilities(model, left, right)
    disp = prob.argmax(dim=0).float() if hard else soft_argmin(prob[None])[0]
    return disp.cpu().numpy()


def evaluate(model, pairs, hard=False):
    """EPE and D1 of the model on full-resolution pairs."""
    return score([predict(model, left, right, hard) for left, right, _ in pairs], pairs)


untrained_pred = predict(model, *val_pairs[0][:2])
untrained_epe, untrained_d1 = evaluate(model, val_pairs)
print(f"Untrained network: mean disparity {untrained_pred.mean():.1f} px | "
      f"validation EPE {untrained_epe:.1f} px, D1 {untrained_d1:.1%}")

## 4. Training

The loss is GC-Net's mean absolute disparity error over the crop pixels with ground truth (the training EPE); disparities of $D$ or more are skipped, since the network cannot output them. We use Adam, as PSMNet does, with the learning rate decaying from $10^{-3}$ to zero along a cosine, and validate every few epochs.

Expected time: about 7 minutes on a T4.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    """One pass over the training crops. Returns the mean L1 loss (the training EPE) in pixels."""
    model.train()
    total_loss, n_batches = 0.0, 0
    # Pass an iterator, not the loader: given a loader, tqdm.auto calls iter() on it once more for
    # itself, which starts an extra set of worker processes every epoch that is never used or shut down.
    progress = tqdm(iter(loader), total=len(loader), desc="train", leave=False)
    for left, right, disp in progress:
        left, right, disp = to_input(left), to_input(right), disp.to(DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(left, right)                       # (B, H, W), float32
        mask = (disp > 0) & (disp < MAX_DISP)
        loss = F.l1_loss(pred[mask], disp[mask])
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                       # the scaler is a no-op when disabled
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches += 1
        progress.set_postfix(loss=f"{loss.item():.2f}")
    return total_loss / n_batches


optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f"{len(train_loader):,} optimizer steps per epoch, {EPOCHS} epochs")

history = {"train_loss": [], "val_epoch": [], "val_epe": [], "val_d1": []}
start = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    scheduler.step()
    history["train_loss"].append(train_loss)
    message = f"Epoch {epoch}/{EPOCHS}: train L1 {train_loss:.3f} px"
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        val_epe, val_d1 = evaluate(model, val_pairs)
        history["val_epoch"].append(epoch)
        history["val_epe"].append(val_epe)
        history["val_d1"].append(val_d1)
        message += f" | val EPE {val_epe:.3f} px, D1 {val_d1:.1%}"
    print(f"{message} | {time.time() - start:.0f}s elapsed")

## 5. Results

The curves, with semi-global matching dashed, then all methods on the validation pairs; the winner-take-all row takes the argmax of the network's probabilities. CPU and GPU times are not comparable.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax_epe, ax_d1) = plt.subplots(1, 2, figsize=(13, 4))
ax_epe.plot(epochs, history["train_loss"], label="training L1 (crops)")
ax_epe.plot(history["val_epoch"], history["val_epe"], marker="o", label="validation EPE")
ax_epe.axhline(sgbm_epe, color="gray", linestyle="--", label="SGBM validation EPE")
ax_epe.set(xlabel="epoch", ylabel="pixels (log scale)", yscale="log", title="End-point error")
ax_epe.legend()
ax_d1.plot(history["val_epoch"], [100 * v for v in history["val_d1"]], marker="o", label="MiniGCNet")
ax_d1.axhline(100 * sgbm_d1, color="gray", linestyle="--", label="SGBM")
ax_d1.set(xlabel="epoch", ylabel="% of ground-truth pixels", title="D1 outliers, validation pairs")
ax_d1.legend()
plt.tight_layout()
plt.show()

start = time.time()
net_preds = [predict(model, left, right) for left, right, _ in tqdm(val_pairs, desc="MiniGCNet")]
net_seconds = (time.time() - start) / len(val_pairs)
net_epe, net_d1 = score(net_preds, val_pairs)
wta_epe, wta_d1 = evaluate(model, val_pairs, hard=True)

print(f"{'method':<34} | {'EPE (px)':>8} | {'D1':>6} | seconds per pair")
print(f"{'untrained network':<34} | {untrained_epe:>8.2f} | {untrained_d1:>6.1%} |")
print(f"{'semi-global matching (CPU)':<34} | {sgbm_epe:>8.2f} | {sgbm_d1:>6.1%} | {sgbm_seconds:.2f}")
print(f"{'MiniGCNet, winner-take-all':<34} | {wta_epe:>8.2f} | {wta_d1:>6.1%} |")
print(f"{'MiniGCNet, soft argmin (' + DEVICE + ')':<34} | {net_epe:>8.2f} | {net_d1:>6.1%} | {net_seconds:.2f}")

Two validation scenes; the bottom rows colour the error at each ground-truth pixel from green (0 px) to red (5 px or more).

In [ ]:
def show_errors(ax, image, pred, gt, title):
    """Ground-truth pixels coloured by absolute disparity error over a dimmed image."""
    ys, xs = np.nonzero(gt > 0)
    ax.imshow(image // 3)
    points = ax.scatter(xs, ys, c=np.minimum(np.abs(pred[ys, xs] - gt[ys, xs]), 5), cmap="RdYlGn_r",
                        vmin=0, vmax=5, s=0.3)
    ax.set_title(title)
    return points


fig, axes = plt.subplots(5, 2, figsize=(16, 13))
for col in range(2):
    left, right, gt = val_pairs[col]
    vmax = np.percentile(gt[gt > 0], 99)
    axes[0, col].imshow(left)
    axes[0, col].set_title(f"Validation scene {val_ids[col]}")
    axes[1, col].imshow(sgbm_preds[col], cmap="magma", vmin=0, vmax=vmax)
    axes[1, col].set_title("Semi-global matching, holes filled")
    axes[2, col].imshow(net_preds[col], cmap="magma", vmin=0, vmax=vmax)
    axes[2, col].set_title("MiniGCNet")
    show_errors(axes[3, col], left, sgbm_preds[col], gt, "Semi-global matching: error")
    points = show_errors(axes[4, col], left, net_preds[col], gt, "MiniGCNet: error")
for ax in axes.flat:
    ax.axis("off")
fig.colorbar(points, ax=axes, fraction=0.02, pad=0.01, label="absolute error (px), bottom two rows")
plt.show()

### Inside the soft argmin

The probabilities $\sigma_d(-\mathbf{c})$ of all 192 candidates at two ground-truth pixels of the first validation scene: an accurate one (the most confident within half a pixel of the truth) and the network's largest error.

In [ ]:
left, right, gt = val_pairs[0]
prob = disparity_probabilities(model, left, right)
peak = prob.max(dim=0).values.cpu().numpy()
ys, xs = np.nonzero(gt > 0)
err = np.abs(net_preds[0] - gt)[ys, xs]
picks = {"an accurate pixel": np.argmax(np.where(err < 0.5, peak[ys, xs], -1.0)),
         "the largest error": np.argmax(err)}

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (name, i) in zip(axes, picks.items()):
    y, x = ys[i], xs[i]
    p = prob[:, y, x].cpu().numpy()
    ax.plot(np.arange(MAX_DISP), p)
    ax.axvline(gt[y, x], color="black", linestyle="--", label=f"ground truth {gt[y, x]:.1f}")
    ax.axvline(net_preds[0][y, x], color="tab:red", label=f"soft argmin {net_preds[0][y, x]:.1f}")
    ax.axvline(p.argmax(), color="tab:orange", linestyle=":", label=f"winner-take-all {p.argmax()}")
    ax.set(xlabel="candidate disparity d (px)", ylabel="probability", title=f"{name} (u={x}, v={y})")
    ax.legend()
plt.tight_layout()
plt.show()

### From disparity to a 3D point cloud

With $Z$ known, pixel $(u, v)$ back-projects to the 3D point $\big((u - c_x) Z / f,\ (v - c_y) Z / f,\ Z\big)$ in the left camera's frame. Seen from above: the network's dense prediction (every second pixel) and the sparse LiDAR ground truth.

In [ ]:
left, _, gt = val_pairs[0]
f, b, cx, cy = CALIBRATION[left.shape[1]]
v, u = np.mgrid[0:left.shape[0], 0:left.shape[1]]


def bird_eye_view(ax, disp, mask, title):
    """Back-project the masked pixels and scatter them in the X-Z plane, coloured by the image."""
    z = f * b / disp[mask]
    x = (u[mask] - cx) * z / f
    near = z < 50
    ax.scatter(x[near], z[near], c=left[mask][near] / 255.0, s=0.3)
    ax.set(xlim=(-20, 20), ylim=(0, 50), xlabel="X, to the right (m)", ylabel="Z, forward (m)", title=title)
    ax.set_aspect("equal")


fig, axes = plt.subplots(1, 2, figsize=(13, 7))
every_second = (u % 2 == 0) & (v % 2 == 0)
bird_eye_view(axes[0], net_preds[0], every_second & (net_preds[0] > 1), "MiniGCNet, dense")
bird_eye_view(axes[1], gt, gt > 0, "LiDAR ground truth, sparse")
plt.tight_layout()
plt.show()

**What to notice**

The training loss often stays high for several epochs, while the network learns only a typical disparity for each part of the image, then falls quickly once its features start to match; the delay varies by run. Semi-global matching is nearly as accurate as the network where it finds a match; most of its outliers lie in the holes we filled, such as the left border. The network wins on both metrics but still errs along car outlines and thin poles, where depth jumps and the quarter-resolution volume blurs the edge. Winner-take-all trails the soft argmin, and far points spread out in depth, as $\Delta Z \approx Z^2 \Delta d / (fB)$ predicts.

We save the weights, reload them into a fresh model and check that the scores match.

In [ ]:
torch.save(model.state_dict(), "mini_gcnet_kitti2015.pth")

reloaded = MiniGCNet(MAX_DISP, FEAT_CH, COST_CH).to(DEVICE)
reloaded.load_state_dict(torch.load("mini_gcnet_kitti2015.pth", map_location=DEVICE))
reloaded_epe, reloaded_d1 = evaluate(reloaded, val_pairs)
print(f"Reloaded model: EPE {reloaded_epe:.3f} px, D1 {reloaded_d1:.1%} (before saving: {net_epe:.3f} px, {net_d1:.1%})")

## Summary

- Rectification makes matching a search along one row, and $Z = fB/d$ turns disparity into depth with an error growing as $Z^2/(fB)$.
- A cost volume scores every candidate disparity of every pixel; semi-global matching hand-designs its cost and smoothing, and leaves holes.
- GC-Net learns the features and regularises a concatenation volume with 3D convolutions, so the matching cost itself is learned.
- The soft argmin is differentiable and sub-pixel, so an L1 loss on sparse LiDAR labels trains the network end to end.

## Exercises

1. **Without the encoder-decoder.** Replace `Hourglass3D(cost_ch)` in `MiniGCNet` with `nn.Identity()` and retrain. How much did its wider context help?
2. **Correlation volume.** Make `build_cost_volume` return `(left_volume * right_volume).mean(dim=1, keepdim=True) * mask`, one dot product per disparity as in DispNetC (Mayer et al., 2016); give the first 3D convolution 1 input channel and retrain. Compare accuracy and epoch time.
3. **Fewer scenes.** Set `N_TRAIN = 40`, `EPOCHS = 120` and `EVAL_EVERY = 20` (the same 1,200 optimizer steps and validation scenes) and retrain. Does the network still beat semi-global matching?
4. **Error with distance.** Group the validation pixels by true depth (0 to 10 m, 10 to 20 m, ...) and compare both methods' disparity and depth errors per group with $Z^2 \Delta d / (fB)$.
5. **Classical settings.** Try `SGBM_BLOCK` of 3 and 9, and `uniquenessRatio=0`. How do EPE, D1 and the rejected share change?

## Further Reading

- [Kendall et al., 2017. End-to-End Learning of Geometry and Context for Deep Stereo Regression](https://arxiv.org/abs/1703.04309)
- [Chang and Chen, 2018. Pyramid Stereo Matching Network](https://arxiv.org/abs/1803.08669)
- [Hirschmüller, 2008. Stereo Processing by Semiglobal Matching and Mutual Information](https://doi.org/10.1109/TPAMI.2007.1166)
- [Menze and Geiger, 2015. Object Scene Flow for Autonomous Vehicles](https://www.cvlibs.net/publications/Menze2015CVPR.pdf)
- [Mayer et al., 2016. A Large Dataset to Train Convolutional Networks for Disparity, Optical Flow, and Scene Flow Estimation](https://arxiv.org/abs/1512.02134)

### Contributed by: Ali Habibullah